2025/04/30 動くか試しに実行
動いた！
５交差検証法を用いた実装。

#References: https://qiita.com/tetsuro731/items/671c81669955d98bb6ff
#概要：LightGBM実行して、テスト結果をMLFlowで確認しよう！

2025/11/16
■運用手順
①Gogle Colabにてmlflowサーバを起動
②起動したURLをmlflow.set_tracking_uri　にセットする
③プログラムを実行する

■変更履歴
1：2026/4/26　モデルを保存するためにpickleを使用

In [3]:
# Colab環境かどうかを判定（ローカルではスキップ）
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("ローカル環境で実行中（Google Driveはマウントされません）")


In [4]:
# Colabの場合のみ、Google Drive内のディレクトリに移動
if IN_COLAB:
    %cd drive/MyDrive/20230614_新機械学習/
else:
    print("ローカル環境: カレントディレクトリを使用")

In [5]:
!pip install lightgbm
!pip install mlflow

In [ ]:
import seaborn as sns
data = sns.load_dataset('titanic')


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import mlflow
import pickle


# Encode categorical features
le = LabelEncoder()
for col in ['sex', 'embarked', 'who', 'embark_town', 'alive']:
    data[col] = le.fit_transform(data[col])

# 特徴量とターゲットに分割
N_splits = 5
train = data.drop('survived', axis=1)
y = data['survived']

#mlflow.lightgbm.autolog() #mlflow実行用
# 環境に応じてMLflowの追跡先を切り替え
if IN_COLAB:
    mlflow.set_tracking_uri("https://24f1-34-75-156-204.ngrok-free.app")  # Colab用: ngrokのURLをセット
else:
    mlflow.set_tracking_uri("file:./mlruns")  # ローカル用: カレントディレクトリのmlrunsを使用
mlflow.set_experiment("colab-demo")  # demo名かえるならメンテ

kfold = KFold(n_splits=N_splits, shuffle=True, random_state=42)
for fold, (trn_ind, val_ind) in enumerate(kfold.split(train)):
  with mlflow.start_run():
    print(' ')
    print('-'*50)
    print(f'Training fold {fold}/{N_splits}....')
    x_train, x_val = train.iloc[trn_ind], train.iloc[val_ind]
    y_train, y_val = y[trn_ind], y[val_ind]

    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'verbosity': -1,
    }
    verbose_eval = 20
    evals_result = {}

    lgb_train = lgb.Dataset(x_train, y_train)
    lgb_valid = lgb.Dataset(x_val, y_val, reference=lgb_train)
    model = lgb.train(
      params = params,
      train_set = lgb_train,
      num_boost_round = 100,
      valid_sets=[lgb_train, lgb_valid],
      callbacks=[lgb.early_stopping(stopping_rounds=20,  verbose=True),
                 lgb.log_evaluation(verbose_eval),
                 lgb.record_evaluation(evals_result)]
    )

    y_pred = model.predict(x_val, num_iteration=model.best_iteration)
    accuracy = accuracy_score(y_val, np.round(y_pred))
    print(f'Fold {fold}, Accuracy: {accuracy}')
    # ※X_trainから1行を抽出
    input_example = pd.DataFrame(x_train.iloc[0]).T  # 最初の1行を入力例として指定

    # モデルをMLflowに保存
    # mlflow.lightgbm.log_model(model, "lightgbm_model", input_example=input_example)

    # パラメータをMLflowに記録
    for param, value in params.items():
        mlflow.log_param(param, value)

    # メトリクスをMLflowに記録
    mlflow.log_metric("best_iteration", model.best_iteration)
    mlflow.log_metric("train_error", evals_result['training']['binary_logloss'][model.best_iteration-1]) # Changed 'train' to 'training' and used best_iteration-1 for index
    mlflow.log_metric("val_error", evals_result['valid_1']['binary_logloss'][model.best_iteration-1]) # Used best_iteration-1 for index
#2026/4/26追加
#モデルを保存するためにpickleを使用
    with open('model.pkl', 'wb') as model_file:
        pickle.dump(model, model_file)

In [15]:
accuracy_score(y_val, np.round(y_pred))

In [18]:
!ls